<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/homework_solutions/hw2_rocket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solution code for HW2 Question 1. Rocket landing trajectory optimization

The solution code reformulates the problem from the 05a notebook with the following:
 1. Added slack variables and penalties on $X_z$, $X_z$, $\theta$ and obstacle constraints
 2. Added $\theta$ constraints
 3. Tuned the cost matrices
 4. Defined a QP problem class for better manipulation and problem setup.
 5. Compared the effects of various trust region penalties on convergence rate




In [ ]:
# !pip install equinox

In [ ]:
import functools
import os
import warnings
from typing import Callable, Dict, List

import cvxpy as cp
import dynamaxsys
import equinox as eqx

# import jax and jax.numpy. No regular numpy needed anymore!
import jax
import jax.numpy as jnp
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

Borrow the plotting helper from 05a notebook.

In [ ]:
# helper functions for plotting
def plot_rocket_trajectory(states, controls, lower, upper, title="Rocket Trajectory"):
    """
    Plots rocket state and control trajectories.

    Args:
        states: Array of shape (n_steps+1, state_dim)
        controls: Array of shape (n_steps, control_dim)
        lower: np.array of shape (control_dim,) — lower limits for controls
        upper: np.array of shape (control_dim,) — upper limits for controls
    """
    n_steps = controls.shape[-2]
    if len(states.shape) == 2:
        alpha = 1.0
    else:
        alpha = 0.4
    fig, axs = plt.subplots(2, 2, figsize=(12, 6), sharex=True)

    # Position
    axs[0, 0].plot(states[..., :, 0].T, color="C0", alpha=alpha)
    axs[0, 0].plot(states[..., :, 1].T, color="C1", alpha=alpha)
    axs[0, 0].set_ylabel("Position (m)")
    leg = axs[0, 0].legend(["p_x (horizontal)", "p_z (altitude)"])
    # Set legend line color to match line color (C0, C1)
    for i, line in enumerate(leg.get_lines()):
        line.set_color(f"C{i}")
    axs[0, 0].grid(True)

    # Angle
    axs[1, 0].plot(states[..., :, 2].T, color="C0", alpha=alpha)
    axs[1, 0].set_ylabel("Angle (rad)")
    axs[1, 0].legend(["theta"])
    axs[1, 0].grid(True)

    # Velocities
    axs[0, 1].plot(states[..., :, 3].T, color="C0", alpha=alpha)
    axs[0, 1].plot(states[..., :, 4].T, color="C1", alpha=alpha)
    axs[0, 1].set_ylabel("Velocity (m/s)")
    leg = axs[0, 1].legend(["v_x", "v_z"])
    for i, line in enumerate(leg.get_lines()):
        line.set_color(f"C{i}")
    axs[0, 1].grid(True)

    # Controls
    axs[1, 1].plot(controls[..., :, 0].T, color="C0", alpha=alpha)
    axs[1, 1].plot(controls[..., :, 1].T, color="C1", alpha=alpha)
    axs[1, 1].plot(controls[..., :, 2].T, color="C2", alpha=alpha)
    leg = axs[1, 1].legend(["T_x", "T_z", "T_nose"])
    for i, line in enumerate(leg.get_lines()):
        line.set_color(f"C{i}")

    axs[1, 1].hlines(lower[0], 0, n_steps, color="C0", linestyle="--")
    axs[1, 1].hlines(upper[0], 0, n_steps, color="C0", linestyle="--")

    axs[1, 1].hlines(lower[1], 0, n_steps, color="C1", linestyle="--")
    axs[1, 1].hlines(upper[1], 0, n_steps, color="C1", linestyle="--")

    axs[1, 1].hlines(lower[2], 0, n_steps, color="C2", linestyle="--")
    axs[1, 1].hlines(upper[2], 0, n_steps, color="C2", linestyle="--")

    axs[1, 1].set_ylabel("Thrust (N)")
    axs[1, 1].set_xlabel("Time step")

    axs[1, 1].grid(True)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_rocket(
    ax: plt.Axes,
    state: jnp.ndarray,
    control: jnp.ndarray | None = None,
    color: str = "C0",
    label: str | None = None,
    rocket_length: float = 2.0,
    thrust_limit: List[float] = [200.0, 300.0, 100.0],
):
    """
    Draw a simple rocket at the specified state, with optional thrust vectors at base and nose.

    Args:
      ax: matplotlib axis
      state: array-like of shape (6,) [x, z, theta, vx, vz, omega]
      control: array-like of shape (,), assumed [thrust, gimbal_angle] or [Tx, Tz], etc.
      color: color for rocket body
      label: optional label for the rocket
      rocket_length: length of the rocket
    """
    x, z, theta = float(state[0]), float(state[1]), float(state[2])

    # Compute end points of rocket body in world frame
    # Rocket body from (-L/2, 0) to (+L/2, 0) in body frame
    dx = rocket_length * np.sin(theta)
    dz = rocket_length * np.cos(theta)

    x_base = x
    z_base = z
    x_nose = x + dx
    z_nose = z + dz

    # Draw main body
    ax.plot(
        [x_base, x_nose],
        [z_base, z_nose],
        color=color,
        linewidth=6,
        solid_capstyle="round",
        label=label,
    )

    # Draw thrust vectors at base and nose if control is provided
    if control is not None:
        T_base_x, T_base_z, T_nose_x = control
        thrust_scale = 5.0  # scaling factor for vector length

        ax.arrow(
            x_base,
            z_base,
            thrust_scale * T_base_x / thrust_limit[0],
            thrust_scale * T_base_z / thrust_limit[1],
            head_width=0.06,
            head_length=0.13,
            fc="crimson",
            ec="crimson",
            alpha=0.6,
            length_includes_head=True,
            zorder=5,
        )
        ax.arrow(
            x_nose,
            z_nose,
            thrust_scale * T_nose_x / thrust_limit[2],
            0,
            head_width=0.06,
            head_length=0.13,
            fc="crimson",
            ec="crimson",
            alpha=0.6,
            length_includes_head=True,
            zorder=5,
        )

    # Draw ground
    ax.hlines(0, -3, 3, color="k", linewidth=2, zorder=0)


def animate_rocket_trajectory(
    states,
    controls=None,
    interval=50,
    ax=None,
    save_path=None,
    color="C0",
    label=None,
    rocket_length=2.0,
    thrust_limit: List[float] = [200.0, 300.0, 100.0],
    obstacle_center: tuple = None,
    obstacle_radius: float = None,
    obstacle_color: str = "gray",
    obstacle_alpha: float = 0.4,
):
    """
    Animate rocket trajectory and pose over time, output as HTML if possible.
    Args:
      states: array-like, shape [N,6]
      controls: array-like, shape [N,...], optional
      interval: ms between frames (default 50)
      ax: optional matplotlib axis
      save_path: optional, if provided save as .mp4, .gif, or .html
      obstacle_center: (x, z) tuple for obstacle center (optional)
      obstacle_radius: float, radius of the obstacle (optional)
      obstacle_color: color spec for obstacle (default 'gray')
      obstacle_alpha: alpha blending value for obstacle (default 0.4)
    """

    states = np.asarray(states)
    if controls is not None:
        controls = np.asarray(controls)
    N = len(states)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 8))
    else:
        fig = ax.figure

    xs = states[:, 0]
    zs = states[:, 1]

    # Draw ground
    ground_x = [xs.min() - 1, xs.max() + 1]
    ground_z = [0, 0]
    ax.plot(ground_x, ground_z, "k-", linewidth=2)

    # Draw circular obstacle if specified
    obstacle_artist = None
    if obstacle_center is not None and obstacle_radius is not None:
        from matplotlib.patches import Circle

        ox, oz = obstacle_center
        obstacle_artist = Circle(
            (ox, oz),
            obstacle_radius,
            color=obstacle_color,
            alpha=obstacle_alpha,
            zorder=1,
        )
        ax.add_patch(obstacle_artist)

    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("z [m]")
    ax.set_title("Rocket landing trajectory")

    # Main trajectory (background)
    (traj_line,) = ax.plot([], [], "k--", alpha=0.4)

    rocket_artists = []

    def init():
        traj_line.set_data([], [])
        for artist in rocket_artists:
            artist.remove()
        rocket_artists.clear()
        return [traj_line] + ([obstacle_artist] if obstacle_artist is not None else [])

    def animate(i):
        # Update trajectory line
        traj_line.set_data(xs[: i + 1], zs[: i + 1])
        # Clear previous rockets
        for artist in rocket_artists:
            artist.remove()
        rocket_artists.clear()
        # Draw rocket at current state
        try:
            plot_rocket(
                ax,
                states[i],
                controls[i],
                color=color,
                label=label,
                rocket_length=rocket_length,
                thrust_limit=thrust_limit,
            )
        except Exception:
            plot_rocket(
                ax,
                states[i],
                color=color,
                label=label,
                rocket_length=rocket_length,
                thrust_limit=thrust_limit,
            )
        # Collect all new artists
        for child in ax.get_children():
            if isinstance(child, (plt.Line2D, plt.Polygon)) and child not in [
                traj_line
            ]:
                rocket_artists.append(child)
        output = [traj_line] + rocket_artists
        if obstacle_artist is not None:
            output.append(obstacle_artist)
        return output

    ani = animation.FuncAnimation(
        fig,
        animate,
        frames=N,
        init_func=init,
        blit=False,
        interval=interval,
        repeat=False,
    )

    # HTML output by default if not saving as a file
    if save_path is not None:
        ext = os.path.splitext(save_path)[1].lower()
        if ext == ".gif":
            ani.save(save_path, writer="pillow")
        display(HTML(ani.to_jshtml()))
        plt.close(fig)
    else:
        display(HTML(ani.to_jshtml()))
        plt.close(fig)


Borrow the simulation helper from 05a notebook.

In [ ]:
# helper functions for simulation
@eqx.filter_jit
def closed_loop_sim(
    dynamics_discrete: dynamaxsys.Dynamics,
    initial_state: jnp.ndarray,
    controller: Callable[[jnp.ndarray], jnp.ndarray],
    sim_steps: int,
):
    """Simulate the closed-loop system given a closed-loop controller using jax.lax.scan"""

    def step_fn(state, _):
        control = controller(state)
        next_state = dynamics_discrete(state, control)
        return next_state, (next_state, control)

    # Dummy scan inputs as we need sim_steps steps
    dummy_inputs = jnp.arange(sim_steps)
    final_state, (all_states, all_controls) = jax.lax.scan(
        step_fn, initial_state, dummy_inputs
    )

    # Include the initial state
    states = jnp.concatenate([initial_state[None], all_states], axis=0)
    return states, all_controls


@eqx.filter_jit
def open_loop_sim(
    dynamics_discrete: dynamaxsys.Dynamics,
    initial_state: jnp.ndarray,
    controls: jnp.ndarray,
):
    """Simulate the open-loop system given a dynamics model using jax.lax.scan"""

    def step_fn(state, control):
        next_state = dynamics_discrete(state, control)
        return next_state, next_state

    # Run scan
    final_state, all_next_states = jax.lax.scan(step_fn, initial_state, controls)

    # Concatenate initial state at the front
    states = jnp.concatenate([initial_state[None], all_next_states], axis=0)
    return states


Borrow the dynamics from 05a notebook.

In [ ]:
class RocketDynamics(dynamaxsys.Dynamics):
    """Rocket dynamics class.
    Refer to 05a-rocket-derivation.md for the derivation of the dynamics.
    """

    def __init__(self, params: Dict[str, float]):
        """
        Initialize the rocket dynamics class.
        Args:
            params: Dictionary containing the rocket parameters.
        """
        state_dim = 6
        control_dim = 3

        def rocket_ode(
            states: jnp.ndarray,
            controls: jnp.ndarray,
            disturbance: jnp.ndarray = None,  # need to provide disturbance for the dynamics class, but not used in this notebook
            time: float = 0.0,  # need to provide time for the dynamics class, but not used in this notebook
        ):
            mass_thruster = params["mass_thruster"]
            mass_body = params["mass_body"]
            length = params["length"]
            gravity = params["gravity"]
            _, _, theta, v_x, v_z, omega = states


            mass = mass_thruster + mass_body

            # First derivatives (kinematic equations)
            velocities = jnp.array([v_x, v_z, omega])

            # Mass matrix M(q)
            M = jnp.array(
                [
                    [mass, 0.0, 0.5 * mass_body * length * jnp.cos(theta)],
                    [0.0, mass, -0.5 * mass_body * length * jnp.sin(theta)],
                    [
                        0.5 * mass_body * length * jnp.cos(theta),
                        -0.5 * mass_body * length * jnp.sin(theta),
                        (1.0 / 3.0) * mass_body * length**2,
                    ],
                ]
            )
            # Coriolis matrix C(q, \dot{q})
            C = jnp.array(
                [
                    [0.0, 0.0, -0.5 * mass_body * length * omega * jnp.sin(theta)],
                    [0.0, 0.0, -0.5 * mass_body * length * omega * jnp.cos(theta)],
                    [0.0, 0.0, 0.0],
                ]
            )

            # gravity vector g(q)
            g_vec = jnp.array(
                [
                    0.0,
                    mass * gravity,
                    -0.5 * mass_body * gravity * length * jnp.sin(theta),
                ]
            )

            # control input Bu with thrust vectoring
            B = jnp.array(
                [
                    [1.0, 0.0, 1.0],
                    [0.0, 1.0, 0.0],
                    [0.0, 0.0, length * jnp.cos(theta)],
                ]
            )
            Bu = B @ controls

            return jnp.concatenate(
                [velocities, jnp.linalg.solve(M, Bu - C @ velocities - g_vec)]
            )


        super().__init__(rocket_ode, state_dim, control_dim)


Borrow the dynamics setting from 05a notebook.

In [ ]:
# feel free to adjust the parameters to see how they affect the rocket's behavior
rocket_length = 1.0  # length of the rocket
params = {
    "mass_thruster": 10.0,  # mass of the thruster
    "mass_body": 15,  # mass of the rocket body
    "length": rocket_length,  # length of the rocket
    "gravity": 9.81,  # gravity constant
}
rocket_ct = RocketDynamics(params)
dt = 0.01
rocket_dt = dynamaxsys.get_discrete_time_dynamics(
    rocket_ct, dt=dt
)  # discrete time dynamics

# set up the control limits
thrust_to_weight_ratio = 1.3
rocket_weight = (params["mass_thruster"] + params["mass_body"]) * params["gravity"]
T_base_z_max = rocket_weight * thrust_to_weight_ratio  # vertical thrust limit
T_base_x_max = T_base_z_max * 0.75  # horizontal thrust limit; feel free to adjust this
T_nose_max = T_base_x_max * 0.75  # nose thrust limit; feel free to adjust this

# set up the lower and upper limits for the control inputs
lower = np.array(
    [-T_base_x_max, 0.0, -T_nose_max]
)  # lower limits for the control inputs
upper = np.array(
    [T_base_x_max, T_base_z_max, T_nose_max]
)  # upper limits for the control inputs
limits = [lower, upper]


Borrow the linearization funciton from 05a notebook.

In [ ]:
@eqx.filter_jit
def linearize_dynamics(
    dynamics: Callable[[jnp.ndarray, jnp.ndarray], jnp.ndarray],
    state: jnp.ndarray,
    control: jnp.ndarray,
    time: float = 0.0,
):
    """Linearize dynamics around the given state and control.

    Args:
        dynamics: Function(state, control, timestep) -> next_state
        state: (n,) array of state around which to linearize
        control: (m,) array of control around which to linearize
        time: float, time at which to linearize, in case dynamics are time-varying

    Returns:
        A: State transition matrix (n, n)
        B: Control input matrix (n, m)
        C: Constant term (n,)
    """


    A, B = jax.jacobian(dynamics, argnums=(0, 1))(state, control, time=time)
    C = dynamics(state, control, dt) - A @ state - B @ control
    return A, B, C



In [ ]:
# obstacle constraint function
def circular_obstacle_constraint(
    state: jnp.ndarray, obs_center: jnp.ndarray, obs_radius: float
) -> jnp.ndarray:
    """Compute obstacle constraint value"""
    p_x, p_z = state[0], state[1]
    obs_x, obs_z = obs_center
    dist_sq = (p_x - obs_x) ** 2 + (p_z - obs_z) ** 2
    return obs_radius**2 - dist_sq


Define the optimization class, so that that we can easily manupulate the optimization variables.

In [ ]:
class RocketSQPProblem:

    def __init__(self,dynamics_ct_in,dynamics_dt_in, lowerin, upperin, theta_lowerin, theta_upperin):
        self.dynamics = dynamics_ct_in
        self.dynamics_dt = dynamics_dt_in
        self.state_dim = dynamics_ct_in.state_dim  # 6, state dimension
        self.control_dim = dynamics_ct_in.control_dim  # 3, control dimension
        self.lower = lowerin
        self.upper = upperin
        self.theta_lower = theta_lowerin
        self.theta_upper = theta_upperin
        self.dynamics_dt = dynamics_dt_in

    def setup_QP(self,Qin=None, Rin=None, QNin=None, QTRPin=None, RTRPin=None,theta_slack_value=None, obstacles_slack_value=None, altitude_slack_value=None, n_steps=None):

        self.Q = cp.Constant(Qin , name="state_cost")  # encourage the rocket to stay upright and slow
        self.R = cp.Constant(Rin , name="control_cost")  # control cost matrix.
        self.QN = cp.Constant(QNin , name="terminal_cost")  # terminal cost matrix.
        self.QTRP = cp.Constant(QTRPin , name="state_trust_region_penalty")  # trust region penalty for state deviations.
        self.RTRP = cp.Constant(RTRPin, name="control_trust_region_penalty")  # trust region penalty for control deviations.

        self.theta_slack_penalty = cp.Constant(theta_slack_value, name="theta_slack_penalty")
        self.obstacles_slack_penalty = cp.Constant(obstacles_slack_value, name="obstacles_slack_penalty")
        self.altitude_slack_penalty = cp.Constant(altitude_slack_value, name="altitude_slack_penalty")



        self.states = cp.Variable((n_steps + 1, self.state_dim))  # state trajectory [n_steps+1, state_dim]
        self.controls = cp.Variable((n_steps, self.control_dim))  # control trajectory [n_steps, control_dim]
        self.obstacle_slack = cp.Variable(n_steps)  # slack variable for obstacle constraint
        self.theta_slack = cp.Variable(n_steps)  # slack variable for angle constraint
        self.altitude_slack = cp.Variable(n_steps)  # slack variable for altitude constraint

    # set up the A, B, C matrices as parameters since they will be updated at each SQP iteration
        self.As = cp.Parameter((n_steps, self.state_dim, self.state_dim), name="As")
        self.Bs = cp.Parameter((n_steps, self.state_dim, self.control_dim), name="Bs")
        self.Cs = cp.Parameter((n_steps, self.state_dim), name="Cs")

    # set up the previous state and control parameters since they will be updated at each SQP iteration
        self.previous_states = cp.Parameter((n_steps + 1, self.state_dim), name="previous_states")
        self.previous_controls = cp.Parameter((n_steps, self.control_dim), name="previous_controls")

    # set up the goal state as a parameter in case you want to change the goal state later on
        self.target_state = cp.Parameter((self.state_dim), name="target_state")

    # set up the initial state parameter in case you want to change the initial state later on
        self.initial_state = cp.Parameter((self.state_dim), name="initial_state")

    # set up control constraints
        self.u_min = cp.Constant(self.lower, name="u_min")
        self.u_max = cp.Constant(self.upper, name="u_max")
        self.theta_min = cp.Constant(self.theta_lower, name="theta_min")  # minimum angle (radians)
        self.theta_max = cp.Constant(self.theta_upper, name="theta_max")  # maximum angle
    # obstacle constraint parameters
        self.Gs = cp.Parameter((n_steps + 1, self.state_dim), name="obstacle_constraint_linear_term")
        self.hs = cp.Parameter((n_steps + 1), name="obstacle_constraint_constant_term")
        self.obstacle_radius = cp.Parameter((), name="obstacle_radius")
        self.obstacle_center = cp.Parameter((2,), name="obstacle_center")
    # update the obstacle center and radius


        self.objective = (self.altitude_slack_penalty * cp.sum_squares(self.altitude_slack) +
                          self.obstacles_slack_penalty * cp.sum_squares(self.obstacle_slack) +
                          self.theta_slack_penalty * cp.sum_squares(self.theta_slack) )

        self.constraints = [self.states[0, :] == self.initial_state, self.obstacle_slack >= 0, self.theta_slack >= 0, self.altitude_slack >= 0]

        for k in range(n_steps):
            self.objective += cp.quad_form(self.controls[k, :], self.R)  # control cost
            self.objective += cp.quad_form(self.states[k, :], self.Q)  # state cost
            self.objective += cp.quad_form(
                self.states[k, :] - self.previous_states[k, :], self.QTRP
            )  # state trust region penalty
            self.objective += cp.quad_form(
                self.controls[k, :] - self.previous_controls[k, :], self.RTRP
            )  # control trust region penalty

            self.constraints += [
                self.states[k + 1, :] == self.As[k] @ self.states[k, :] + self.Bs[k] @ self.controls[k, :] + self.Cs[k]
            ]  # dynamics constraint
            self.constraints += [
                self.Gs[k] @ self.states[k, :] + self.hs[k] <= self.obstacle_slack[k]
            ]  # obstacle constraint
            self.constraints += [self.controls[k, :] >= self.u_min ]  # control lower bound
            self.constraints += [self.controls[k, :] <= self.u_max ]  # control upper bound

            self.constraints += [
                self.states[k,1] >= -self.altitude_slack[k]
            ]  # negative vertical position


            self.constraints += [
                self.states[k, 2] <= self.theta_max + self.theta_slack[k]
            ]  # angle constraint
            self.constraints += [
                self.states[k, 2] >= 0 - self.theta_slack[k]
            ]  # angle constraint

        self.objective += cp.quad_form(self.states[n_steps, :] - self.target_state, self.QN)  # terminal state cost
        self.constraints += [self.Gs[-1] @ self.states[-1, :] + self.hs[-1] <= self.obstacle_slack[-1]]  # terminal state constraint
        self.constraints += [self.states[n_steps, 1] >= -self.altitude_slack[-1]]  # terminal state constraint

        self.prob = cp.Problem(cp.Minimize(self.objective), self.constraints)

In [ ]:
# Initialize three instances of the RocketSQPProblem class for
# the three different trust region penalty settings

rocket_W1 = RocketSQPProblem(dynamics_ct_in = rocket_ct, dynamics_dt_in = rocket_dt, lowerin = lower, upperin = upper, theta_lowerin = -np.pi/4, theta_upperin = np.pi/4)
rocket_W2 = RocketSQPProblem(dynamics_ct_in = rocket_ct, dynamics_dt_in = rocket_dt, lowerin = lower, upperin = upper, theta_lowerin = -np.pi/4, theta_upperin = np.pi/4)
rocket_W3 = RocketSQPProblem(dynamics_ct_in = rocket_ct, dynamics_dt_in = rocket_dt, lowerin = lower, upperin = upper, theta_lowerin = -np.pi/4, theta_upperin = np.pi/4)

In [ ]:
state_dim = rocket_ct.state_dim  # 6, state dimension
control_dim = rocket_ct.control_dim  # 3, control dimension
n_steps=300

# set up the cost matrices and slack penalties.
Q = np.diag([1.0, 0, 20.0, 0.5, 0.5, 10.0])
R = np.diag([0.005, 0.001, 0.0])
QN = np.diag([50.0, 500.0, 10000000.0, 1000.0, 800.0, 1000.0]) * 100# terminal cost matrix.
theta_slack_penalty = 10000.0
obstacles_slack_penalty = 100000000.0
altitude_slack_penalty = 10000.0


# define three different trust region penalties.
QTRP_w1 = np.eye(state_dim) * 0.00 # trust region penalty for state deviations.
RTRP_w1 = np.eye(control_dim) * 0.00 # trust region penalty for control deviations.

QTRP_w2 = np.eye(state_dim) * 5.0
RTRP_w2 = np.eye(control_dim) * 5.0

QTRP_w3 = np.eye(state_dim) * 9.0
RTRP_w3 = np.eye(control_dim) * 9.0

# set up the QP for each instance with the corresponding trust region penalties
rocket_W1.setup_QP(Qin=Q, Rin=R, QNin=QN, QTRPin=QTRP_w1, RTRPin=RTRP_w1, theta_slack_value=theta_slack_penalty, obstacles_slack_value=obstacles_slack_penalty, altitude_slack_value=altitude_slack_penalty, n_steps=n_steps)
rocket_W2.setup_QP(Qin=Q, Rin=R, QNin=QN, QTRPin=QTRP_w2, RTRPin=RTRP_w2, theta_slack_value=theta_slack_penalty, obstacles_slack_value=obstacles_slack_penalty, altitude_slack_value=altitude_slack_penalty, n_steps=n_steps)
rocket_W3.setup_QP(Qin=Q, Rin=R, QNin=QN, QTRPin=QTRP_w3, RTRPin=RTRP_w3, theta_slack_value=theta_slack_penalty, obstacles_slack_value=obstacles_slack_penalty, altitude_slack_value=altitude_slack_penalty, n_steps=n_steps)

In [ ]:
# set up the initial guess for the state and control trajectories.

# target state value
target_state_value = np.array(
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
)  # feel free to adjust target state

# choose obstacle center and radius
obstacle_center_value = np.array([-0.01, 4.0])  # feel free to adjust obstacle center
obstacle_radius_value = np.array(0.5)  # feel free to adjust obstacle radius

# obstacle constraint function. Use functools.partial to fix the obstacle center and radius.
obstacle_constraint = functools.partial(
    circular_obstacle_constraint,
    obs_center=obstacle_center_value,
    obs_radius=obstacle_radius_value,
)

# initial state value
initial_state_value = np.array([0.1, 8.0, 0.1, -0.1, -0.1, -0.05])

controls_values_int = np.zeros([n_steps, control_dim])
states_values_int = open_loop_sim(
    rocket_dt, initial_state_value, controls_values_int
)



# update the initial guess for the state and control trajectories for each instance
rocket_W1.target_state.value = target_state_value
rocket_W1.initial_state.value = initial_state_value

rocket_W2.target_state.value = target_state_value
rocket_W2.initial_state.value = initial_state_value

rocket_W3.target_state.value = target_state_value
rocket_W3.initial_state.value = initial_state_value

# update the obstacle center and radius
rocket_W1.obstacle_center.value = obstacle_center_value
rocket_W1.obstacle_radius.value = obstacle_radius_value

rocket_W2.obstacle_center.value = obstacle_center_value
rocket_W2.obstacle_radius.value = obstacle_radius_value

rocket_W3.obstacle_center.value = obstacle_center_value
rocket_W3.obstacle_radius.value = obstacle_radius_value


In [ ]:
# Define a function to run the SQP iterations.

def solveRocketSQP(rocket_sqp_problem, n_SQP_iterations,states_int, controls_int, obstacle_constraint):
    controls_list = [controls_int]
    costs = []
    previous_states_values = states_int
    previous_controls_values = controls_int
    for i in range(n_SQP_iterations):
        print(f"SQP Iteration {i + 1}/{n_SQP_iterations}")

        rocket_sqp_problem.previous_states.value = np.array(previous_states_values)
        rocket_sqp_problem.previous_controls.value = np.array(previous_controls_values)

        As_values, Bs_values, Cs_values = jax.vmap(
            linearize_dynamics, in_axes=(None, 0, 0)
        )(rocket_sqp_problem.dynamics_dt, previous_states_values[:-1], previous_controls_values)

        Gs_values = jax.vmap(jax.jacobian(obstacle_constraint, argnums=0), in_axes=(0,))(
            previous_states_values
        )
        hs_values = jax.vmap(obstacle_constraint)(previous_states_values) - jax.vmap(
            jnp.dot, in_axes=(0, 0)
        )(Gs_values, previous_states_values)

        rocket_sqp_problem.As.value = np.array(As_values)
        rocket_sqp_problem.Bs.value = np.array(Bs_values)
        rocket_sqp_problem.Cs.value = np.array(Cs_values)
        rocket_sqp_problem.Gs.value = np.array(Gs_values)
        rocket_sqp_problem.hs.value = np.array(hs_values)

            # solve the problem

        rocket_sqp_problem.prob.solve(solver=cp.CLARABEL)
        costs.append(rocket_sqp_problem.objective.value)

            # update the previous states and controls
        previous_states_values = rocket_sqp_problem.states.value
        previous_controls_values = rocket_sqp_problem.controls.value

        #open_loop_states = open_loop_sim(rocket_dt, initial_state_value, rocket_sqp_problem.controls.value)
        controls_list.append(rocket_sqp_problem.controls.value)

    costs_list = np.array(costs)
    controls_list = np.array(controls_list)

    return costs_list  , controls_list



In [ ]:
n_SQP_iterations = 100

# run the SQP solver for each instance and collect the costs and controls for each iteration
costs_list_w1, controls_list_w1 = solveRocketSQP(rocket_W1, n_SQP_iterations, states_values_int, controls_values_int, obstacle_constraint)
costs_list_w2, controls_list_w2 = solveRocketSQP(rocket_W2, n_SQP_iterations, states_values_int, controls_values_int, obstacle_constraint)
costs_list_w3, controls_list_w3 = solveRocketSQP(rocket_W3, n_SQP_iterations, states_values_int, controls_values_int, obstacle_constraint)

# simulate the trajectories for each instance
# using the open-loop dynamics and the controls obtained from the SQP solver
states_list_w1 = jax.vmap(open_loop_sim, in_axes=(None, None, 0))(
    rocket_dt, initial_state_value, controls_list_w1
)
states_list_w2 = jax.vmap(open_loop_sim, in_axes=(None, None, 0))(
    rocket_dt, initial_state_value, controls_list_w2
)
states_list_w3 = jax.vmap(open_loop_sim, in_axes=(None, None, 0))(
    rocket_dt, initial_state_value, controls_list_w3
)


In [ ]:
# Plot the cost vs. SQP iteration for each instance
plt.plot(costs_list_w1)
plt.plot(costs_list_w2)
plt.plot(costs_list_w3)
plt.xlabel("SQP iteration")
plt.ylabel("Cost")
plt.title("Cost vs SQP iteration for different trust region penalties")
plt.legend(["trust_region_penalty=0.01", "trust_region_penalty=1.0", "trust_region_penalty=5.0"])
plt.grid(True)
plt.xlim([0, 20])


In [ ]:
# Plot the cost vs. SQP iteration in log scale

plt.plot(np.log(costs_list_w1))
plt.plot(np.log(costs_list_w2))
plt.plot(np.log(costs_list_w3))
plt.xlabel("SQP iteration")
plt.ylabel("Cost (log scale)")
plt.title("Cost in Log Scale vs SQP iteration for different trust region penalties")
plt.grid(True)
plt.legend(["trust_region_penalty=0.01", "trust_region_penalty=1.0", "trust_region_penalty=5.0"])

In [ ]:
# plot the trajectories for the instance with the trust region penalty setting of 5.0
# at each iteration
plot_rocket_trajectory(states_list_w3, controls_list_w3, lower, upper)

In [ ]:
# Plot the final trajectory for the instance with trust region penalty = 5.0
plot_rocket_trajectory(states_list_w3[-1], controls_list_w3[-1], lower, upper, "Trust region penalty = 5.0")

In [ ]:
animate_rocket_trajectory(
    states_list_w3[-1],
    controls_list_w3[-1],
    obstacle_center=obstacle_center_value,
    obstacle_radius=obstacle_radius_value,
    thrust_limit=upper,
)

In [ ]:
# Plot the final trajectory for the instance with trust region penalty = 5.0
plot_rocket_trajectory(states_list_w1, controls_list_w1, lower, upper)